In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal

from utils import get_resource_metadata_field, find_distributions, find_api_endpoints, get_rowstore_data, NAMESPACE

[EntryStore Documentation](https://entrystore.org)

[EntryStore API Documentation](https://entrystore.org/api)

### Examples of GET:

`{base}{context-id}/metadata/{entry-id}`

Catalog Metadata: https://zg-demo.entryscape.net/store/1/metadata/1

Dataset Metadata: https://zg-demo.entryscape.net/store/1/metadata/5

Distribution metadata: https://zg-demo.entryscape.net/store/1/metadata/7

Define IDs and catalog URL.

In [ ]:
context_id = "1"
dataset_id = "5"
distribution_id = "7"

catalog_url = f"https://zg-demo.entryscape.net/store/{context_id}"

Using rdflib, parse the rdf-graph of a dataset and print it:

In [ ]:
g = Graph()

# Parse dataset RDF
dataset_url = f"{catalog_url}/resource/{dataset_id}"

g.parse(dataset_url)

g.print(format="turtle")

In [ ]:
g = Graph()
distribution_id = "9"
# Parse dataset RDF
distribution_url = f"{catalog_url}/resource/{distribution_id}"

g.parse(distribution_url)

g.print(format="json-ld")

In [ ]:
DCAT = Namespace(NAMESPACE["dcat"])
resource_ref = URIRef(f"{catalog_url}/resource/{distribution_id}")

values = [o.toPython() if isinstance(o, Literal) else str(o) for o in g.objects(resource_ref, DCAT["accessURL"])]
values

Get the value of a single metadata field, e.g., `dcterms:title`

In [ ]:
get_resource_metadata_field(
    catalog_url,
    "7",
    "dcterms:title"
)

Find resources with format `application/json`

In [ ]:
find_distributions(
    catalog_url,
    dataset_id,
    format_mime="application/json"
)

Find resources with api endpoint

In [ ]:
api_endpoints = find_api_endpoints(
    catalog_url,
    dataset_id
)

Get data from RowStore API

Query parameter can be regex

In [ ]:
api_endpoint = api_endpoints[0]

query_params = {
    "jahr": "202[0-3]",
    "typ": "^(?!Personenwagen$).*"
}

data = get_rowstore_data(
    api_endpoint,
    query_params,
    fetch_all=True
)

In [ ]:
import pandas as pd
df = pd.DataFrame(data)
# convert the col anzahl to numeric
df['anzahl'] = pd.to_numeric(df['anzahl'], errors='coerce')
df

Visualization

In [ ]:
import plotly.express as px

fig = px.line(
    df,
    x="jahr",
    y="anzahl",
    color="typ",
    labels={"jahr": "Jahr", "anzahl": "Anzahl"},
    )

fig.show()